In [2]:
!pip install pandas requests openpyxl rapidfuzz folium geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.5/32.5 MB 80.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 7.7 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 4.2 MB/s eta 0:00:00ta 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import requests
import pandas as pd
import re
import os
from io import BytesIO, StringIO
from datetime import datetime

# ----------------------------------------------------------------------
# Column cleaning and standardization (exact same logic as original)
# ----------------------------------------------------------------------
def clean_and_standardize(df):
    """Standardize column names and enforce a uniform schema."""
    df = df.copy()
    df.columns = df.columns.str.strip()

    column_mappings = {
        'Date': ['Report Date', 'Date', 'Incident Date', 'Date & Time'],
        'Time': ['Time', 'Incident Time'],
        'Day': ['Day'],
        'Location': ['Location', 'Station', 'Station Name', 'Stop', 'Stop Name'],
        'Incident': ['Incident', 'Code', 'Description'],
        'Min Delay': ['Min Delay', 'Delay', 'Delay Minutes', 'Delay_Minutes'],
        'Min Gap': ['Min Gap', 'Gap', 'Gap Minutes', 'Gap_Minutes'],
        'Route': ['Route', 'Route Number', 'Route No', 'Route_ID'],
        'Line': ['Line'],
        'Direction': ['Direction', 'Bound'],
        'Vehicle': ['Vehicle', 'Vehicle Number', 'Vehicle_No']
    }

    reverse_mapping = {}
    for std_name, possible in column_mappings.items():
        for name in possible:
            reverse_mapping[name] = std_name

    rename_dict = {col: reverse_mapping[col] for col in df.columns if col in reverse_mapping}
    df = df.rename(columns=rename_dict)

    # Handle Line column -> Route, Route Name
    if 'Line' in df.columns and 'Route' not in df.columns:
        def extract_route_info(line_val):
            if pd.isna(line_val):
                return pd.Series([None, None])
            line_str = str(line_val).strip()
            match = re.match(r'^(\d+)(?:\s+(.+))?$', line_str)
            if match:
                return pd.Series([match.group(1), match.group(2) if match.group(2) else ''])
            match_digits = re.match(r'^(\d+)$', line_str)
            if match_digits:
                return pd.Series([match_digits.group(1), ''])
            return pd.Series([line_str, ''])
        df[['Route', 'Route Name']] = df['Line'].apply(extract_route_info)

    if 'Route' in df.columns and 'Route Name' not in df.columns:
        df['Route Name'] = ''

    required_columns = [
        'Date', 'Route', 'Route Name', 'Time', 'Day', 'Location',
        'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle',
        'Incident_Original'
    ]
    for col in required_columns:
        if col not in df.columns:
            df[col] = None

    df = df[required_columns]
    return df


# ----------------------------------------------------------------------
# Download data for a single mode
# ----------------------------------------------------------------------
def load_mode_delay_data(mode_name, package_id, extra_csv=None, force_csv_only=False, skip_year_filter=False):
    """Downloads all resources for a given transit mode, cleans and returns a DataFrame."""
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    current_year = datetime.now().year
    print(f"\n🚋 Processing {mode_name} data (package: {package_id})")

    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception(f"CKAN API request failed for {mode_name}")

    resources = data["result"]["resources"]
    print(f"📦 Found {len(resources)} resource(s)")

    year_pattern = re.compile(r"(19|20)\d{2}")
    allowed_formats = {'csv'} if force_csv_only else {'csv', 'xlsx', 'xls'}

    mode_dfs = []

    for res in resources:
        res_name = res.get("name", "Unnamed")
        datastore_active = res.get("datastore_active", False)

        if not skip_year_filter:
            year_match = year_pattern.search(res_name)
            if not year_match:
                print(f"  ⏭️  {res_name}: no year found, skipping")
                continue
            year = int(year_match.group(0))
            if year < 2014:
                print(f"  ⏭️  {res_name}: year {year} < 2014, skipping")
                continue
        else:
            year = None

        # Determine format
        if datastore_active:
            fmt = 'csv'
        else:
            fmt = res.get("format", "").lower()
            if not fmt:
                url = res.get("url", "")
                if url.endswith('.csv'):
                    fmt = 'csv'
                elif url.endswith(('.xlsx', '.xls')):
                    fmt = 'xlsx'
                else:
                    fmt = 'unknown'

        if fmt not in allowed_formats:
            print(f"  ⏭️  {res_name}: format '{fmt}' not in {allowed_formats}, skipping")
            continue

        # Apply year‑based format filter only if not skipped
        if not skip_year_filter and not force_csv_only:
            if year >= current_year-1 and fmt != 'csv':
                print(f"  ⏭️  {res_name}: skipping XLSX for latest year, only CSV kept")
                continue
            elif year < current_year-1 and fmt not in ('xlsx', 'xls'):
                print(f"  ⏭️  {res_name}: skipping CSV for year < {current_year}, only XLSX kept")
                continue

        if year:
            print(f"  📄 Resource: {res_name} (year={year}, format={fmt}, datastore_active={datastore_active})")
        else:
            print(f"  📄 Resource: {res_name} (format={fmt}, datastore_active={datastore_active})")

        try:
            if datastore_active:
                dump_url = f"{base_url}/datastore/dump/{res['id']}"
                dump_resp = requests.get(dump_url)
                dump_resp.raise_for_status()
                df = pd.read_csv(StringIO(dump_resp.text))
                if 'Incident' in df.columns:
                    df['Incident_Original'] = df['Incident']
            else:
                file_url = res["url"]
                file_resp = requests.get(file_url)
                file_resp.raise_for_status()

                if fmt in ('xlsx', 'xls'):
                    excel_data = pd.read_excel(BytesIO(file_resp.content), sheet_name=None)
                    sheet_dfs = []
                    for sheet_name, sheet_df in excel_data.items():
                        if not sheet_df.empty:
                            if 'Incident' in sheet_df.columns:
                                sheet_df['Incident_Original'] = sheet_df['Incident']
                            sheet_df = clean_and_standardize(sheet_df)
                            sheet_dfs.append(sheet_df)
                    if sheet_dfs:
                        df = pd.concat(sheet_dfs, ignore_index=True)
                    else:
                        print(f"    ⚠️ No data in any sheet, skipping")
                        continue
                else:
                    df = pd.read_csv(StringIO(file_resp.text))
                    if 'Incident' in df.columns:
                        df['Incident_Original'] = df['Incident']

            # Standardize columns (keeps Incident_Original and original Incident)
            df = clean_and_standardize(df)

            # Add transit mode column (will be kept in final output)
            df['Transit'] = mode_name

            mode_dfs.append(df)
            print(f"    ✅ Loaded {len(df)} records")

        except Exception as e:
            print(f"    ❌ Error: {e}")
            continue

    if extra_csv and os.path.isfile(extra_csv):
        print(f"\n  📄 Extra local file: {extra_csv}")
        try:
            df_extra = pd.read_csv(extra_csv)
            if 'Incident' in df_extra.columns:
                df_extra['Incident_Original'] = df_extra['Incident']
            df_extra = clean_and_standardize(df_extra)
            df_extra['Transit'] = mode_name
            mode_dfs.append(df_extra)
            print(f"    ✅ Loaded {len(df_extra)} records from extra file")
        except Exception as e:
            print(f"    ❌ Error reading extra file {extra_csv}: {e}")
    elif extra_csv:
        print(f"\n  ⏭️ Extra file {extra_csv} not found, skipping")

    if not mode_dfs:
        print(f"⚠️ No valid data loaded for {mode_name}")
        return pd.DataFrame()

    mode_combined = pd.concat(mode_dfs, ignore_index=True)
    print(f"✅ {mode_name} total records: {len(mode_combined)}")
    return mode_combined


# ----------------------------------------------------------------------
# Main function: download and merge all modes, return filtered DataFrame
# with original standardized columns + Transit, and no nulls in Route/Date/MinDelay (MinDelay > 0)
# ----------------------------------------------------------------------
def get_clean_ttc_delays():
    """
    Download all TTC delay datasets (Bus, Streetcar, Subway, LRT),
    merge them, filter rows where Route, Date, and Min Delay are not null
    and Min Delay > 0, and return only the original standardized columns plus Transit.
    """
    modes = [
        ("Bus", "ttc-bus-delay-data"),
        ("Streetcar", "ttc-streetcar-delay-data"),
        ("Subway", "ttc-subway-delay-data"),
        ("LRT", "ttc-lrt-delay-data")
    ]

    all_dfs = []
    for mode_name, pkg_id in modes:
        # For LRT, download all resources without year filtering (as in original)
        if mode_name == "LRT":
            df_mode = load_mode_delay_data(mode_name, pkg_id,
                                           extra_csv="TTC LRT Delays.csv",
                                           force_csv_only=False,
                                           skip_year_filter=True)
        else:
            df_mode = load_mode_delay_data(mode_name, pkg_id)
        if not df_mode.empty:
            all_dfs.append(df_mode)

    if not all_dfs:
        raise Exception("No data loaded for any mode.")

    merged_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n📊 Total merged records (before filtering): {len(merged_df)}")

    # Drop duplicates
    initial_len = len(merged_df)
    merged_df = merged_df.drop_duplicates()
    print(f"🧹 Removed {initial_len - len(merged_df)} duplicate rows")

    # Keep only the original standardized columns plus Transit.
    # Original standardized columns: Date, Time, Day, Location, Incident,
    # Min Delay, Min Gap, Route, Direction, Vehicle.
    # We also keep Transit.
    cols_to_keep = ['Date', 'Time', 'Day', 'Location', 'Incident',
                    'Min Delay', 'Min Gap', 'Route', 'Direction', 'Vehicle',
                    'Transit']
    # Ensure only columns that actually exist are selected (some may be missing in some datasets)
    cols_to_keep = [col for col in cols_to_keep if col in merged_df.columns]
    merged_df = merged_df[cols_to_keep]

    # Filter rows: Route, Date, Min Delay not null, and Min Delay > 0
    before_filter = len(merged_df)
    merged_df = merged_df.dropna(subset=['Route', 'Date', 'Min Delay'])
    merged_df = merged_df[merged_df['Min Delay'] > 0]
    print(f"🔍 Kept {len(merged_df)} rows after filtering (removed {before_filter - len(merged_df)} rows with null Route/Date/MinDelay or MinDelay <= 0)")

    return merged_df


# ----------------------------------------------------------------------
# Example usage
# ----------------------------------------------------------------------
if __name__ == "__main__":
    df = get_clean_ttc_delays()
    print("\n👀 First few rows (original columns + Transit):")
    print(df.head())
    print("\n📋 Columns returned:", list(df.columns))


🚋 Processing Bus data (package: ttc-bus-delay-data)
📦 Found 20 resource(s)
  ⏭️  ttc-bus-delay-data-readme: no year found, skipping
  📄 Resource: ttc-bus-delay-data-2014 (year=2014, format=xlsx, datastore_active=False)
    ✅ Loaded 94217 records
  📄 Resource: ttc-bus-delay-data-2015 (year=2015, format=xlsx, datastore_active=False)
    ✅ Loaded 76510 records
  📄 Resource: ttc-bus-delay-data-2016 (year=2016, format=xlsx, datastore_active=False)
    ✅ Loaded 77088 records
  📄 Resource: ttc-bus-delay-data-2017 (year=2017, format=xlsx, datastore_active=False)
    ✅ Loaded 70303 records
  📄 Resource: ttc-bus-delay-data-2018 (year=2018, format=xlsx, datastore_active=False)
    ✅ Loaded 73927 records
  📄 Resource: ttc-bus-delay-data-2019 (year=2019, format=xlsx, datastore_active=False)
    ✅ Loaded 62376 records
  📄 Resource: ttc-bus-delay-data-2020 (year=2020, format=xlsx, datastore_active=False)
    ✅ Loaded 36151 records
  📄 Resource: ttc-bus-delay-data-2021 (year=2021, format=xlsx, datast

In [4]:
df_all = df.copy()

In [5]:
# 1. Convert 'Date' to datetime (coerce errors to NaT)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# 2. Extract useful components from the date
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Weekday'] = df['Date'].dt.day_name()

# 3. Extract hour from the 'Time' column (handles "HH:MM" and "HH:MM:SS")
def extract_hour(time_val):
    if pd.isna(time_val):
        return None
    try:
        # Split by ':' and take the first part (hour)
        hour = int(str(time_val).split(':')[0])
        return hour
    except (ValueError, AttributeError):
        return None

df['Hour'] = df['Time'].apply(extract_hour)

# 4. Convert Location column to all uppercase
df['Location'] = df['Location'].str.upper()

In [6]:
# ----------------------------------------------------------------------
# Define the incident code mapping (from the original pipeline)
# ----------------------------------------------------------------------
def _build_code_mapping():
    """Build dictionary mapping raw incident codes to standard categories."""
    mapping = {}

    # ---- Subway codes (EU, MU, PU, SU, TU) ----
    # Equipment / Mechanical (EU)
    eu_mech = [
        'EUAC', 'EUAL', 'EUATC', 'EUBK', 'EUBO', 'EUCA', 'EUCH', 'EUCO',
        'EUDO', 'EUECD', 'EUHV', 'EULT', 'EULV', 'EUNEA', 'EUNT', 'EUO',
        'EUPI', 'EUSC', 'EUTL', 'EUTM', 'EUTR', 'EUTRD', 'EUVA', 'EUVE', 'EUYRD'
    ]
    for code in eu_mech:
        mapping[code] = 'Equipment / Mechanical'

    mapping['EUCD'] = 'General Delay / Other'          # Consequential Delay
    mapping['EUME'] = 'Operations / Human Error'       # Maintenance Error
    mapping['EUOE'] = 'Operations / Human Error'       # Rail Cars & Shops Opr. Error
    mapping['EUOPO'] = 'Infrastructure / Track / Signals'

    # Miscellaneous (MU)
    mapping['MUD']   = 'Passenger / Security'
    mapping['MUDD']  = 'External / Environment'
    mapping['MUEC']  = 'Infrastructure / Track / Signals'
    mapping['MUESA'] = 'Operations / Human Error'
    mapping['MUFM']  = 'External / Environment'
    mapping['MUFS']  = 'External / Environment'
    mapping['MUGD']  = 'General Delay / Other'
    mapping['MUI']   = 'Passenger / Security'
    mapping['MUIE']  = 'Passenger / Security'
    mapping['MUIR']  = 'Passenger / Security'
    mapping['MUIRS'] = 'Passenger / Security'
    mapping['MUIS']  = 'Passenger / Security'
    mapping['MULD']  = 'Management / Administrative'
    mapping['MUNOA'] = 'Operations / Human Error'
    mapping['MUO']   = 'General Delay / Other'
    mapping['MUODC'] = 'Infrastructure / Track / Signals'
    mapping['MUPAA'] = 'Passenger / Security'
    mapping['MUPLA'] = 'External / Environment'
    mapping['MUPLB'] = 'External / Environment'
    mapping['MUPLC'] = 'External / Environment'
    mapping['MUPR1'] = 'Passenger / Security'
    mapping['MUSAN'] = 'Cleaning / Unsanitary'
    mapping['MUSC']  = 'Equipment / Mechanical'
    mapping['MUTD']  = 'Management / Administrative'
    mapping['MUTO']  = 'General Delay / Other'
    mapping['MUWEA'] = 'External / Environment'
    mapping['MUWR']  = 'Management / Administrative'

    # Infrastructure (PU)
    pu_infra = [
        'PUATC', 'PUCBI', 'PUCSC', 'PUCSS', 'PUDCS', 'PUMEL', 'PUMO',
        'PUOPO', 'PUSAC', 'PUSBE', 'PUSCA', 'PUSCR', 'PUSEA', 'PUSI',
        'PUSIO', 'PUSIS', 'PUSLC', 'PUSO', 'PUSRA', 'PUSSW', 'PUSTC',
        'PUSTP', 'PUSTS', 'PUSWZ', 'PUSZC', 'PUTCD', 'PUTD', 'PUTIJ',
        'PUTNT', 'PUTO', 'PUTOE', 'PUTR', 'PUTS', 'PUTSC', 'PUTSM',
        'PUTTC', 'PUTTP', 'PUTWZ'
    ]
    for code in pu_infra:
        mapping[code] = 'Infrastructure / Track / Signals'

    mapping['PUMST'] = 'Passenger / Security'
    mapping['PUTDN'] = 'External / Environment'
    mapping['PUTIS'] = 'External / Environment'
    mapping['PUSNT'] = 'General Delay / Other'

    # Security (SU)
    su_sec = [
        'SUAE', 'SUAP', 'SUBT', 'SUCOL', 'SUDP', 'SUEAS', 'SUG',
        'SUO', 'SUPOL', 'SUROB', 'SUSA', 'SUSP', 'SUUT'
    ]
    for code in su_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TU)
    mapping['TUATC'] = 'Operations / Human Error'
    mapping['TUCC']  = 'Operations / Human Error'
    mapping['TUDOE'] = 'Operations / Human Error'
    mapping['TUKEY'] = 'Operations / Human Error'
    mapping['TUML']  = 'Scheduling / Late Starts'
    mapping['TUMVS'] = 'Operations / Human Error'
    mapping['TUNIP'] = 'Operations / Human Error'
    mapping['TUNOA'] = 'Operations / Human Error'
    mapping['TUO']   = 'General Delay / Other'
    mapping['TUOPO'] = 'Operations / Human Error'
    mapping['TUOS']  = 'Operations / Human Error'
    mapping['TUS']   = 'Scheduling / Late Starts'
    mapping['TUSC']  = 'Operations / Human Error'
    mapping['TUSET'] = 'Operations / Human Error'
    mapping['TUST']  = 'External / Environment'
    mapping['TUSUP'] = 'Operations / Human Error'

    # ---- Streetcar codes (ER, MR, PR, SR, TR) ----
    # Equipment / Mechanical (ER)
    er_mech = [
        'ERAC', 'ERBO', 'ERCO', 'ERDB', 'ERDO', 'ERHV', 'ERLT', 'ERLV',
        'ERNEA', 'ERNT', 'ERO', 'ERPR', 'ERRA', 'ERTB', 'ERTC', 'ERTL',
        'ERTR', 'ERVE', 'ERWA', 'ERWS'
    ]
    for code in er_mech:
        mapping[code] = 'Equipment / Mechanical'
    mapping['ERCD'] = 'General Delay / Other'
    mapping['ERME'] = 'Operations / Human Error'

    # Miscellaneous (MR)
    mapping['MRCL']  = 'Management / Administrative'
    mapping['MRD']   = 'Passenger / Security'
    mapping['MRDD']  = 'External / Environment'
    mapping['MREC']  = 'Infrastructure / Track / Signals'
    mapping['MRESA'] = 'Operations / Human Error'
    mapping['MRFS']  = 'External / Environment'
    mapping['MRIE']  = 'Passenger / Security'
    mapping['MRLD']  = 'Management / Administrative'
    mapping['MRNOA'] = 'Operations / Human Error'
    mapping['MRO']   = 'General Delay / Other'
    mapping['MRPAA'] = 'Passenger / Security'
    mapping['MRPLA'] = 'External / Environment'
    mapping['MRPLB'] = 'External / Environment'
    mapping['MRPLC'] = 'External / Environment'
    mapping['MRPR1'] = 'Passenger / Security'
    mapping['MRSAN'] = 'Cleaning / Unsanitary'
    mapping['MRSTM'] = 'Infrastructure / Track / Signals'
    mapping['MRTO']  = 'General Delay / Other'
    mapping['MRUI']  = 'Passenger / Security'
    mapping['MRUIR'] = 'Passenger / Security'
    mapping['MRWEA'] = 'External / Environment'

    # Infrastructure (PR)
    mapping['PREL']  = 'Infrastructure / Track / Signals'
    mapping['PRO']   = 'General Delay / Other'
    mapping['PRS']   = 'Infrastructure / Track / Signals'
    mapping['PRSA']  = 'Infrastructure / Track / Signals'
    mapping['PRSL']  = 'Infrastructure / Track / Signals'
    mapping['PRSO']  = 'Infrastructure / Track / Signals'
    mapping['PRSP']  = 'Infrastructure / Track / Signals'
    mapping['PRST']  = 'Passenger / Security'
    mapping['PRSW']  = 'Infrastructure / Track / Signals'
    mapping['PRTST'] = 'Infrastructure / Track / Signals'
    mapping['PRW']   = 'Infrastructure / Track / Signals'

    # Security (SR)
    sr_sec = [
        'SRAE', 'SRAP', 'SRBT', 'SRCOL', 'SRDP', 'SREAS', 'SRO',
        'SRSA', 'SRSP', 'SRUT'
    ]
    for code in sr_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TR)
    mapping['TRDOE'] = 'Operations / Human Error'
    mapping['TRNIP'] = 'Operations / Human Error'
    mapping['TRNOA'] = 'Operations / Human Error'
    mapping['TRO']   = 'General Delay / Other'
    mapping['TRSET'] = 'Operations / Human Error'
    mapping['TRST']  = 'External / Environment'
    mapping['TRTC']  = 'Operations / Human Error'

    # ---- LRT codes (EX, MX, PX, SX, TX) ----
    # Equipment / Mechanical (EX)
    ex_mech = [
        'EXAC', 'EXBK', 'EXBO', 'EXCB', 'EXCE', 'EXCO', 'EXDB', 'EXDO',
        'EXECD', 'EXGA', 'EXGF', 'EXHV', 'EXLT', 'EXNEA', 'EXNT', 'EXO',
        'EXOSC', 'EXSA', 'EXSE', 'EXTB', 'EXTM', 'EXTR', 'EXVC', 'EXVE',
        'EXWA', 'EXWM', 'EXWS', 'EXYRD'
    ]
    for code in ex_mech:
        mapping[code] = 'Equipment / Mechanical'
    mapping['EXADD'] = 'Equipment / Mechanical'
    mapping['EXOE']  = 'Operations / Human Error'
    mapping['EXPD']  = 'Collision / Roadblock'
    mapping['EXPI']  = 'Collision / Roadblock'

    # Miscellaneous (MX)
    mapping['MXAFR'] = 'Infrastructure / Track / Signals'
    mapping['MXCL']  = 'Management / Administrative'
    mapping['MXCSA'] = 'Operations / Human Error'
    mapping['MXD']   = 'Passenger / Security'
    mapping['MXDD']  = 'External / Environment'
    mapping['MXESA'] = 'Operations / Human Error'
    mapping['MXFM']  = 'External / Environment'
    mapping['MXFS']  = 'External / Environment'
    mapping['MXGD']  = 'General Delay / Other'
    mapping['MXI']   = 'Passenger / Security'
    mapping['MXIC']  = 'Passenger / Security'
    mapping['MXIE']  = 'Passenger / Security'
    mapping['MXIR']  = 'Passenger / Security'
    mapping['MXIRS'] = 'Passenger / Security'
    mapping['MXIS']  = 'Passenger / Security'
    mapping['MXLDC'] = 'Management / Administrative'
    mapping['MXLDT'] = 'Management / Administrative'
    mapping['MXNCA'] = 'Operations / Human Error'
    mapping['MXNOA'] = 'Operations / Human Error'
    mapping['MXO']   = 'General Delay / Other'
    mapping['MXPAA'] = 'Passenger / Security'
    mapping['MXPF']  = 'Infrastructure / Track / Signals'
    mapping['MXPLA'] = 'External / Environment'
    mapping['MXPLB'] = 'External / Environment'
    mapping['MXPLC'] = 'External / Environment'
    mapping['MXPR']  = 'External / Environment'
    mapping['MXPR1'] = 'Passenger / Security'
    mapping['MXPU']  = 'Operations / Human Error'
    mapping['MXSAN'] = 'Cleaning / Unsanitary'
    mapping['MXTD']  = 'Management / Administrative'
    mapping['MXTO']  = 'General Delay / Other'
    mapping['MXUS']  = 'Scheduling / Late Starts'
    mapping['MXWEA'] = 'External / Environment'
    mapping['MXWR']  = 'Management / Administrative'

    # Infrastructure (PX)
    mapping['PXATC'] = 'Infrastructure / Track / Signals'
    mapping['PXDCS'] = 'Infrastructure / Track / Signals'
    mapping['PXEAS'] = 'Infrastructure / Track / Signals'
    mapping['PXEME'] = 'Operations / Human Error'
    mapping['PXEO']  = 'General Delay / Other'
    mapping['PXMEL'] = 'Infrastructure / Track / Signals'
    mapping['PXMO']  = 'General Delay / Other'
    mapping['PXMST'] = 'Passenger / Security'
    mapping['PXOV']  = 'Infrastructure / Track / Signals'
    mapping['PXSAC'] = 'Infrastructure / Track / Signals'
    mapping['PXSBE'] = 'Infrastructure / Track / Signals'
    mapping['PXSCA'] = 'Infrastructure / Track / Signals'
    mapping['PXSCR'] = 'Infrastructure / Track / Signals'
    mapping['PXSI']  = 'Infrastructure / Track / Signals'
    mapping['PXSIS'] = 'Infrastructure / Track / Signals'
    mapping['PXSNT'] = 'General Delay / Other'
    mapping['PXSO']  = 'General Delay / Other'
    mapping['PXSRA'] = 'Infrastructure / Track / Signals'
    mapping['PXSTP'] = 'Infrastructure / Track / Signals'
    mapping['PXSW']  = 'Infrastructure / Track / Signals'
    mapping['PXTD']  = 'Infrastructure / Track / Signals'
    mapping['PXTDN'] = 'External / Environment'
    mapping['PXTIS'] = 'External / Environment'
    mapping['PXTR']  = 'Infrastructure / Track / Signals'
    mapping['PXTS']  = 'Infrastructure / Track / Signals'
    mapping['PXW']   = 'Infrastructure / Track / Signals'
    mapping['PXWZ']  = 'Infrastructure / Track / Signals'

    # Security (SX)
    sx_sec = [
        'SXAE', 'SXAM', 'SXAP', 'SXAX', 'SXBT', 'SXCOL', 'SXDP',
        'SXEAS', 'SXG', 'SXO', 'SXPOL', 'SXROB', 'SXSA', 'SXSP', 'SXUEG'
    ]
    for code in sx_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TX)
    mapping['TXATC'] = 'Operations / Human Error'
    mapping['TXCC']  = 'Operations / Human Error'
    mapping['TXDOE'] = 'Operations / Human Error'
    mapping['TXLF']  = 'Scheduling / Late Starts'
    mapping['TXML']  = 'Scheduling / Late Starts'
    mapping['TXMVS'] = 'Operations / Human Error'
    mapping['TXNCA'] = 'Operations / Human Error'
    mapping['TXNIP'] = 'Operations / Human Error'
    mapping['TXNOA'] = 'Operations / Human Error'
    mapping['TXO']   = 'General Delay / Other'
    mapping['TXOI']  = 'Passenger / Security'
    mapping['TXOS']  = 'Operations / Human Error'
    mapping['TXOVS'] = 'Operations / Human Error'
    mapping['TXPD']  = 'Collision / Roadblock'
    mapping['TXPI']  = 'Collision / Roadblock'
    mapping['TXS']   = 'Scheduling / Late Starts'
    mapping['TXST']  = 'External / Environment'
    mapping['TXSUP'] = 'Operations / Human Error'
    mapping['TXSV']  = 'Operations / Human Error'

    return mapping

# Global mapping dictionary
_INCIDENT_CODE_MAP = _build_code_mapping()

# ----------------------------------------------------------------------
# Category name translation (remove slashes, use single words)
# ----------------------------------------------------------------------
CATEGORY_TRANSLATION = {
    'Equipment / Mechanical': 'Mechanical',
    'Operations / Human Error': 'Operations',
    'Infrastructure / Track / Signals': 'Infrastructure',
    'Passenger / Security': 'Passenger',
    'External / Environment': 'External',
    'Scheduling / Late Starts': 'Scheduling',
    'Collision / Roadblock': 'Collision',
    'Cleaning / Unsanitary': 'Cleaning',
    'Management / Administrative': 'Management',
    'General Delay / Other': 'General'
}

# ----------------------------------------------------------------------
# Function to map any incident value to a clean category
# ----------------------------------------------------------------------
def map_to_category(incident):
    """
    Convert an incident code or description into one of ten categories:
    Mechanical, Operations, Infrastructure, Passenger, External,
    Scheduling, Collision, Cleaning, Management, General.
    """
    if pd.isna(incident):
        return 'General'
    s = str(incident).strip()
    if not s:
        return 'General'

    # Step 1: If it looks like a pure code (all caps, 2‑6 letters), use the code map
    if re.match(r'^[A-Z]{2,6}$', s):
        old_cat = _INCIDENT_CODE_MAP.get(s)
        if old_cat:
            return CATEGORY_TRANSLATION.get(old_cat, 'General')
        else:
            return 'General'

    # Step 2: Direct matches for common plain‑text values
    direct_map = {
        'Mechanical': 'Mechanical',
        'General Delay': 'General',
        'Emergency Services': 'Passenger',
        'Investigation': 'Passenger',
        'Diversion': 'Operations',
        'Late Leaving Garage': 'Scheduling',
        'Utilized Off Route': 'Operations',
        'Vision': 'Operations',
        'Late Leaving Garage - Operator': 'Operations',
        'Late Leaving Garage - Mechanical': 'Mechanical',
        'Late Leaving Garage - Management': 'Management',
        'Late Leaving Garage - Vision': 'Operations',
        'Management': 'Management',
        'Operations - Operator': 'Operations',
        'Cleaning': 'Cleaning',
        'Security': 'Passenger',
        'Collision - TTC': 'Collision',
        'Road Blocked - NON-TTC Collision': 'Collision',
        'Road Block - Non-TTC Collision': 'Collision',
        'Roadblock by Collision - Non-TTC': 'Collision',
        'Securitty': 'Passenger',
        'Late Entering Service - Mechanical': 'Mechanical',
        'Held By': 'Passenger',
        'Late Leaving Garage - Operations': 'Operations',
        'e': 'General',
        'Late Entering Service': 'Scheduling',
        'Operations': 'Operations',
        'Cleaning - Unsanitary': 'Cleaning',
        'Cleaning - Disinfection': 'Cleaning',
        'Collision - TTC Involved': 'Collision',
        'Late': 'Scheduling',
        'Overhead': 'Infrastructure',
        'Rail/Switches': 'Infrastructure',
        'MFESA': 'Operations',
        'MFSAN': 'Cleaning',
        'MFUI': 'Passenger',
        'EFO': 'General',
        'EFP': 'General',
        'TFCNO': 'Operations',
        'SFDP': 'Passenger',
        'MFDV': 'Operations',
        'SFPOL': 'Passenger',
        'EFHVA': 'Mechanical',
        'MFUIR': 'Passenger',
        'TFO': 'General',
        'MFO': 'General',
        'MFVIS': 'Operations',
        'EFD': 'General',
        'EFB': 'General',
        'TFPD': 'Collision',
        'TFOI': 'Passenger',
        'MFTO': 'General',
        'MFSH': 'General',
        'MFPI': 'Collision',
        'EFRA': 'Mechanical',
        'MFUS': 'Scheduling',
        'SFO': 'General',
        'MFS': 'General',
        'SFAE': 'Passenger',
        'SFAP': 'Passenger',
        'MFPR': 'External',
        'SFSA': 'Passenger',
        'TFPI': 'Collision',
        'MFWEA': 'External',
        'MTO': 'General',
        'EFCAN': 'General',
        'MTVIS': 'Operations',
        'MTNOA': 'Operations',
        'MTDV': 'Operations',
        'MFFD': 'Passenger',
        'TTPD': 'Collision',
        'STO': 'General',
        'MUIS': 'Passenger',
        'PFPD': 'Collision',
        'MTUS': 'Scheduling',
        'MTSAN': 'Cleaning',
        'MUO': 'General',
        'PFO': 'General',
        'TFLF': 'Scheduling',
        'TFLL': 'Scheduling',
        'ETO': 'General',
        'Overhead - Pantograph': 'Infrastructure',
        'Late  ': 'Scheduling',
        'MTAFR': 'Infrastructure',
        'MTIE': 'Passenger',
        'MTUIR': 'Passenger',
        'ETVC': 'Mechanical',
        'MTUI': 'Passenger',
        'TTSW': 'Operations',
        'STDP': 'Passenger',
        'TTOI': 'Passenger',
        'TTO': 'General',
        'ETRA': 'Mechanical',
        'ETDB': 'Mechanical',
        'TTUS': 'Operations',
        'ETPI': 'Mechanical',
        'PTOV': 'Infrastructure',
        'MTTP': 'Infrastructure',
        'MTPU': 'Operations',
        'STAP': 'Passenger',
        'MTESA': 'Operations',
        'ETDO': 'Mechanical',
        'MTTO': 'General',
        'MTGD': 'General',
        'ETSA': 'Mechanical',
        'ETHV': 'Mechanical',
        'TTPI': 'Collision',
        'ETAC': 'Mechanical',
        'STAE': 'Passenger',
        'PTO': 'Infrastructure',
        'MTPI': 'Collision',
        'ETWS': 'Mechanical',
        'MTPOL': 'Passenger',
        'ETBO': 'Mechanical',
        'ETCE': 'Mechanical',
        'ETTR': 'Mechanical',
        'MTWEA': 'External',
        'ETWA': 'Mechanical',
        'ETSE': 'Mechanical',
        'ETLT': 'Mechanical',
        'PTSW': 'Infrastructure',
        'ETCM': 'Mechanical',
        'MTS': 'External',
        'ETVE': 'Mechanical',
        'ETNT': 'General',
        'PTSE': 'Infrastructure',
        'ETFA': 'Mechanical',
        'ETCO': 'Mechanical',
        'ETLV': 'Mechanical',
        'ETAX': 'Mechanical',
        'STSA': 'Passenger',
        'SUDP': 'Passenger',
        'ETTM': 'Mechanical',
        'ETTB': 'Mechanical',
        'MTTD': 'Management',
        'MTEC': 'Infrastructure',
        'SUO': 'Passenger',
        'TTLL': 'Scheduling',
        'STSP': 'Passenger',
        'TTLF': 'Scheduling',
        'ETDS': 'Mechanical',
        'MUPR1': 'Passenger',
        'MUSAN': 'Cleaning',
        'MUNOA': 'Operations',
        'MUTO': 'General',
        'SUEAS': 'Passenger',
        'MUIR': 'Passenger',
        'MUI': 'Passenger',
        'MUPLB': 'External',
        'SUUT': 'Passenger',
        'EUDO': 'Mechanical',
        'ERWA': 'Mechanical',
        'MUDD': 'External',
        'TUNOA': 'Operations',
        'ERDO': 'Mechanical',
        'MUSC': 'Mechanical',
        'EUPI': 'Mechanical',
        'PUTIS': 'External',
        'TUNIP': 'Operations',
        'ERTC': 'Mechanical',
        'MUPAA': 'Passenger',
        'PUSI': 'Infrastructure',
        'PUSIS': 'Infrastructure',
        'SUG': 'Passenger',
        'EUECD': 'Mechanical',
        'TUML': 'Scheduling',
        'MUD': 'Passenger',
        'EUNT': 'General',
        'EUNEA': 'General',
        'EUCH': 'Mechanical',
        'PUTTC': 'Infrastructure',
        'SUAE': 'Passenger',
        'SUAP': 'Passenger',
        'PUSO': 'Infrastructure',
        'TUO': 'General',
        'ERPR': 'Mechanical',
        'PUCSS': 'Infrastructure',
        'TUSC': 'Operations',
        'PUTIJ': 'Infrastructure',
        'PUSTC': 'Infrastructure',
        'TUST': 'External',
        'MUNCA': 'Operations',
        'ERTL': 'Mechanical',
        'EUME': 'Operations',
        'MRTO': 'General',
        'TUOS': 'Operations',
        'PUSTS': 'Infrastructure',
        'EUTRD': 'Mechanical',
        'TUMVS': 'Operations',
        'MUWEA': 'External',
        'MRWEA': 'External',
        'PUSNT': 'General',
        'SUSA': 'Passenger',
        'SUBT': 'Passenger',
        'TUSUP': 'Operations',
        'EUBK': 'Mechanical',
        'EUSC': 'Mechanical',
        'SUPOL': 'Passenger',
        'EULV': 'Mechanical',
        'PUTWZ': 'Infrastructure',
        'MRO': 'General',
        'TUDOE': 'Operations',
        'MUTD': 'Management',
        'MRNOA': 'Operations',
        'MRUI': 'Passenger',
        'ERCO': 'Mechanical',
        'ERNEA': 'General',
        'MRUIR': 'Passenger',
        'ERLV': 'Mechanical',
        'EUCA': 'Mechanical',
        'TUCC': 'Operations',
        'TUS': 'Scheduling',
        'PUCSC': 'Infrastructure',
        'EUAC': 'Mechanical',
        'EUCD': 'General',
        'EUBO': 'Mechanical',
        'MUPLA': 'External',
        'EUTR': 'Mechanical',
        'ERDB': 'Mechanical',
        'EUVE': 'Mechanical',
        'PUTDN': 'External',
        'MUWR': 'Management',
        'EUVA': 'Mechanical',
        'EUOE': 'Operations',
        'TUSET': 'Operations',
        'EUO': 'General',
        'PUTTP': 'Infrastructure',
        'TUKEY': 'Operations',
        'EUCO': 'Mechanical',
        'ERNT': 'General',
        'PRO': 'General',
        'TRO': 'General',
        'TRNOA': 'Operations',
        'MUEC': 'Infrastructure',
        'MRPLA': 'External',
        'PUTSM': 'Infrastructure',
        'ERRA': 'Mechanical',
        'PUSCA': 'Infrastructure',
        'PUSSW': 'Infrastructure',
        'SUROB': 'Passenger',
        'MRPLB': 'External',
        'SRCOL': 'Passenger',
        'SUCOL': 'Passenger',
        'PUSCR': 'Infrastructure',
        'MRPAA': 'Passenger',
        'PRW': 'Infrastructure',
        'PUTD': 'Infrastructure',
        'TRNIP': 'Operations',
        'TRTC': 'Operations',
        'EUTL': 'Mechanical',
        'ERBO': 'Mechanical',
        'ERTO': 'General',
        'PUTR': 'Infrastructure',
        'EUYRD': 'Mechanical',
        'PUTO': 'Infrastructure',
        'SRAP': 'Passenger',
        'PUTSC': 'Infrastructure',
        'SRUT': 'Passenger',
        'EULT': 'Mechanical',
        'MUESA': 'Operations',
        'EUAL': 'Mechanical',
        'SRDP': 'Passenger',
        'PUSWZ': 'Infrastructure',
        'PRS': 'Infrastructure',
        'SREAS': 'Passenger',
        'ERAC': 'Mechanical',
        'PUSRA': 'Infrastructure',
        'ERTB': 'Mechanical',
        'PUMO': 'Infrastructure',
        'SRO': 'Passenger',
        'MUFM': 'External',
        'MUPLC': 'External',
        'PUMEL': 'Infrastructure',
        'PUTS': 'Infrastructure',
        'TUTD': 'Management',
        'TRSET': 'Operations',
        'EUHV': 'Mechanical',
        'EUTM': 'Mechanical',
        'PUTOE': 'Infrastructure',
        'PUSTP': 'Infrastructure',
        'PUMST': 'Passenger',
        'MUIRS': 'Passenger',
        'ERLT': 'Mechanical',
        'PREL': 'Infrastructure',
        'SUSP': 'Passenger',
        'ERTR': 'Mechanical',
        'ERO': 'General',
        'TRST': 'External',
        'ERME': 'Operations',
        'ERHV': 'Mechanical',
        'ERVE': 'Mechanical',
        'SRBT': 'Passenger',
        'PUTNT': 'General',
        'PUTCD': 'Infrastructure',
        'TRDOE': 'Operations',
        'EUOPO': 'Infrastructure',
        'PUOPO': 'Infrastructure',
        'PUSEA': 'Infrastructure',
        'MRD': 'Passenger',
        'MUCL': 'Management',
        'PRSL': 'Infrastructure',
        'PRTST': 'Infrastructure',
        'MRDD': 'External',
        'MUIE': 'Passenger',
        'MRCL': 'Management',
        'PRSW': 'Infrastructure',
        'PRSA': 'Infrastructure',
        'MRSAN': 'Cleaning',
        'MUGD': 'General',
        'TUOPO': 'Operations',
        'MUATC': 'Infrastructure',
        'PUATC': 'Infrastructure',
        'MUFS': 'External',
        'EUATC': 'Infrastructure',
        'TUATC': 'Operations',
        'MRIE': 'Passenger',
        'PRSO': 'Infrastructure',
        'PUSZC': 'Infrastructure',
        'SRAE': 'Passenger',
        'PUEO': 'Infrastructure',
        'MRESA': 'Operations',
        'PUEWZ': 'Infrastructure',
        'MUPF': 'Infrastructure',
        'PUSAC': 'Infrastructure',
        'ERCD': 'General',
        'ERWS': 'Mechanical',
        'PUSIO': 'Infrastructure',
        'PUEME': 'Operations',
        'PRSP': 'Infrastructure',
        'TUNCA': 'Operations',
        'SUPD': 'Passenger',
        'MRPR1': 'Passenger',
        'EUTAC': 'Mechanical',
        'MRSTM': 'Infrastructure',
        'MRPLC': 'External',
        'MRFS': 'External',
        'TUUR': 'Operations',
        'MUCP': 'Infrastructure',
        'EXTM': 'Mechanical',
        'EXBK': 'Mechanical',
        'EXBO': 'Mechanical',
        'MXTO': 'General',
        'EXO': 'General',
        'TXO': 'General',
        'TXDOE': 'Operations',
        'TXOS': 'Operations',
        'MXUS': 'Scheduling',
        'PXWZ': 'Infrastructure',
        'TXNIP': 'Operations',
        'MXO': 'General',
        'EXDO': 'Mechanical',
        'EXAC': 'Mechanical',
        'PXSTP': 'Infrastructure',
        'PXSW': 'Infrastructure',
        'PXTIS': 'External',
        'MXAFR': 'Infrastructure',
        'PXSIS': 'Infrastructure',
        'EXNEA': 'General',
        'EXCE': 'Mechanical',
        'TXCC': 'Operations',
        'EXATC': 'Mechanical',
        'EXSA': 'Mechanical',
        'EXECD': 'Mechanical',
        'EXGA': 'Mechanical',
        'MXIE': 'Passenger',
        'MXPAA': 'Passenger',
        'PXTDN': 'External',
        'TXOVS': 'Operations',
        'EXYRD': 'Mechanical',
        'TXS': 'Scheduling',
        'PXTS': 'Infrastructure',
        'SXUEG': 'Passenger',
        'MXIR': 'Passenger',
        'PXDCS': 'Infrastructure',
        'EXTR': 'Mechanical',
        'TXSUP': 'Operations',
        'TXNOA': 'Operations',
        'EXWS': 'Mechanical',
        'MXNOA': 'Operations',
        'TXCL': 'Management',
        'MXWEA': 'External',
        'MXIRS': 'Passenger',
        'MXPLC': 'External',
        'EXWA': 'Mechanical',
        'MXD': 'Passenger',
    }

    if s in direct_map:
        return direct_map[s]

    # Step 3: Keyword‑based pattern matching (same logic as original)
    s_lower = s.lower()
    patterns = [
        (r'late (leaving|entering)|unable to maintain schedule|mainline storage', 'Scheduling'),
        (r'collision|road ?block', 'Collision'),
        (r'clean|unsanitary|disinfection', 'Cleaning'),
        (r'management|clerk|training|labour dispute|work refusal', 'Management'),
        (r'weather|ice|snow|fire|debris|force majeure|storm', 'External'),
        (r'mechanical|equipment|brakes|door.*faulty|hvac|propulsion', 'Mechanical'),
        (r'operations?.*operator|signal violation|overshot|overspeed|not in position|supervisory', 'Operations'),
        (r'passenger|security|assault|disorderly|bomb|alarm|unauthorized|injur', 'Passenger'),
        (r'infrastructure|track|signal|power|escalator|elevator|switch|rail|debris.*controllable', 'Infrastructure'),
    ]
    for pattern, category in patterns:
        if re.search(pattern, s_lower):
            return category

    return 'General'

# ----------------------------------------------------------------------
# Apply the mapping to create the new column
# ----------------------------------------------------------------------
df['Incident_Category'] = df['Incident'].apply(map_to_category)

In [7]:
import re

def clean_route(val):
    """
    Convert route values to integer route numbers:
    - Direct replacements: YU→1, BD→2, SRT→3, SHP→985, FW→6
    - Otherwise, extract the first integer found in the string.
    - If no integer found, return None.
    """
    if pd.isna(val):
        return None
    s = str(val).strip()
    
    # Direct replacements for subway line codes
    replacement_map = {
        'YU': 1,
        'BD': 2,
        'SRT': 3,
        'SHP': 985,
        'FW': 6
    }
    if s in replacement_map:
        return replacement_map[s]
    
    # Extract first integer (including those in strings like "32.0")
    match = re.search(r'\d+', s)
    if match:
        return int(match.group())
    
    # No number found – leave as None (or you could keep original string)
    return None

# Apply to the Route column (updates in place)
df['Route'] = df['Route'].apply(clean_route)
df = df[df['Route'].notnull()]

In [ ]:
import requests
import pandas as pd
import zipfile
from io import BytesIO, StringIO

# ----------------------------------------------------------------------
# Function to download routes.txt only
# ----------------------------------------------------------------------
def download_routes_gtfs():
    """Download routes.txt from the TTC GTFS package and return as DataFrame."""
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"

    # Get package metadata
    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # Find the ZIP resource (complete GTFS)
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # Extract routes.txt
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        if 'routes.txt' in zf.namelist():
            with zf.open('routes.txt') as f:
                routes_df = pd.read_csv(f)
            print("✅ Extracted routes.txt")
        else:
            raise Exception("routes.txt not found in ZIP")

    return routes_df

# ----------------------------------------------------------------------
# Download routes.txt
# ----------------------------------------------------------------------
routes_df = download_routes_gtfs()

# Display a preview
print("\n📋 routes.txt columns:", list(routes_df.columns))
print(routes_df[['route_id', 'route_short_name', 'route_long_name']].head())

# ----------------------------------------------------------------------
# Prepare for merging
# ----------------------------------------------------------------------
# Ensure both key columns are clean
# Convert Route to integer where possible (to match route_short_name)
# First, drop rows where Route is missing
df = df.dropna(subset=['Route'])

# Convert Route to integer (if it's a float like 95.0, it becomes 95)
df['Route'] = pd.to_numeric(df['Route'], errors='coerce').astype('Int64')

# Convert route_short_name to integer as well (GTFS stores them as strings but they are numeric)
routes_df['route_short_name'] = pd.to_numeric(routes_df['route_short_name'], errors='coerce').astype('Int64')

# Drop rows where route_short_name is missing (just in case)
routes_df = routes_df.dropna(subset=['route_short_name'])

# ----------------------------------------------------------------------
# Debug: Check sample values
# ----------------------------------------------------------------------
print("\nSample values from df['Route'] (first 20):")
print(df['Route'].dropna().head(20).tolist())
print("\nSample values from routes_df['route_short_name'] (first 20):")
print(routes_df['route_short_name'].head(20).tolist())

# ----------------------------------------------------------------------
# Remove any existing 'route_long_name' column to avoid merge conflicts
# ----------------------------------------------------------------------
if 'route_long_name' in df.columns:
    print("⚠️ Removing existing 'route_long_name' column before merge.")
    df = df.drop(columns=['route_long_name'])

# ----------------------------------------------------------------------
# Left join to add route_long_name
# ----------------------------------------------------------------------
# Keep only necessary columns from routes_df to avoid duplication
df = df.merge(routes_df[['route_short_name', 'route_long_name']],
              left_on='Route',
              right_on='route_short_name',
              how='left')

# Drop the temporary key column
df = df.drop(columns=['route_short_name'])

# ----------------------------------------------------------------------
# Check results
# ----------------------------------------------------------------------
print("\n✅ Route names added. Columns now:", list(df.columns))
print("\nSample matches:")
print(df[['Route', 'route_long_name']].drop_duplicates().head(10))

# Count how many rows got a match vs. missing
matched = df['route_long_name'].notna().sum()
total = len(df)
print(f"\n📊 Matched {matched} out of {total} rows ({matched/total*100:.1f}%)")

📥 Downloading GTFS ZIP from: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/b811ead4-6eaf-4adb-8408-d389fb5a069c/resource/c920e221-7a1c-488b-8c5b-6d8cd4e85eaf/download/completegtfs.zip
✅ Extracted routes.txt

📋 routes.txt columns: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_url', 'route_color', 'route_text_color']
   route_id  route_short_name       route_long_name
0        10                10             Van Horne
1       100               100       Flemingdon Park
2       101               101        Downsview Park
3       102               102            Markham Rd
4       103               103  Mount Pleasant North

✅ Route names added. Sample matches:
    Route route_long_name
0    95.0             NaN
1   102.0             NaN
2    54.0             NaN
3   112.0             NaN
4    24.0             NaN
5   129.0             NaN
6    36.0             NaN
7    53.0             NaN
9   320.0             NaN
10   91.0

In [9]:
# ----------------------------------------------------------------------
# Ensure Route is integer (clean up any floats or strings)
# ----------------------------------------------------------------------
# Convert to numeric, coerce errors to NaN, then to nullable integer
df['Route'] = pd.to_numeric(df['Route'], errors='coerce').astype('Int64')

# ----------------------------------------------------------------------
# Manual route name mapping (for routes missing in GTFS or needing override)
# ----------------------------------------------------------------------
route_name_manual = {
    3: "Line 3 (Scarborough RT)",
    56: "Leaside",
    195: "Jane Rocket",
    196: "York University Express",
    199: "Finch Rocket",
    # Add others if needed
}

# ----------------------------------------------------------------------
# Fill missing route_long_name using manual map
# ----------------------------------------------------------------------
mask = df['route_long_name'].isna() & df['Route'].isin(route_name_manual.keys())
df.loc[mask, 'route_long_name'] = df.loc[mask, 'Route'].map(route_name_manual)

# ----------------------------------------------------------------------
# Check results
# ----------------------------------------------------------------------
print("Updated route_long_name for the specified routes:")
print(df[df['Route'].isin([3, 56, 195, 196, 199])][['Route', 'route_long_name']].drop_duplicates())

Updated route_long_name for the specified routes:
       Route          route_long_name
49       199             Finch Rocket
189      196  York University Express
663       56                  Leaside
8518       3  Line 3 (Scarborough RT)
25473    195              Jane Rocket


In [10]:
import pandas as pd
df = pd.read_csv(r'C:\Users\bains\OneDrive\Documents\VS Code\Toronto-Transit-Delays\assets\data\full_delay_data.csv')

In [10]:
route_counts = df['Route'].value_counts()

# Identify routes with at least 10 records
routes_to_keep = route_counts[route_counts >= 10].index

# Filter the DataFrame
initial_rows = len(df)
df = df[df['Route'].isin(routes_to_keep)]

# Report
removed_rows = initial_rows - len(df)
print(f"Removed {removed_rows} rows from routes with < 10 records.")
print(f"Remaining rows: {len(df)}")

Removed 406 rows from routes with < 10 records.
Remaining rows: 973808


In [11]:
if 'Route' in df.columns:
    df['Route'] = pd.to_numeric(df['Route'], errors='coerce').astype('Int64')

# Define route groups
subway_routes = [1, 2, 3, 4]
lrt_routes = [5, 6]
streetcar_routes = [301, 304, 305, 306, 310, 312,
                    501, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512]

# Create a condition for each group (only where Route is not null)
subway_mask = df['Route'].isin(subway_routes)
lrt_mask = df['Route'].isin(lrt_routes)
streetcar_mask = df['Route'].isin(streetcar_routes)

# Update Transit (overwrites previous values)
df.loc[subway_mask, 'Transit'] = 'Subway'
df.loc[lrt_mask, 'Transit'] = 'LRT'
df.loc[streetcar_mask, 'Transit'] = 'Streetcar'

# All remaining numeric routes (not in any special list) become Bus
# Also, if Route is null, keep original Transit? Usually we leave as is.
# But we'll set to Bus for any numeric route not covered.
numeric_mask = df['Route'].notna()
bus_mask = numeric_mask & ~(subway_mask | lrt_mask | streetcar_mask)
df.loc[bus_mask, 'Transit'] = 'Bus'

In [ ]:
import requests
import pandas as pd
import zipfile
from io import BytesIO, StringIO

# ----------------------------------------------------------------------
# Function to download trips.txt only
# ----------------------------------------------------------------------
def download_trips_gtfs():
    """Download trips.txt from the TTC GTFS package and return as DataFrame."""
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"

    # Get package metadata
    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # Find the ZIP resource
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # Extract trips.txt
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        if 'trips.txt' in zf.namelist():
            with zf.open('trips.txt') as f:
                trips_df = pd.read_csv(f)
            print("✅ Extracted trips.txt")
        else:
            raise Exception("trips.txt not found in ZIP")

    return trips_df

# ----------------------------------------------------------------------
# Download trips.txt and build variant mapping
# ----------------------------------------------------------------------
trips_df = download_trips_gtfs()
print(f"Trips loaded: {len(trips_df)} rows")

# Clean columns
trips_df['route_id'] = trips_df['route_id'].astype(str).str.strip()
trips_df['trip_short_name'] = trips_df['trip_short_name'].fillna('').astype(str).str.strip()

# Keep only rows with non‑empty trip_short_name (these define variants)
variants_df = trips_df[trips_df['trip_short_name'] != '']

# Group by route_id and collect unique trip_short_names
variant_map = variants_df.groupby('route_id')['trip_short_name'].unique().reset_index()
variant_map['trip_short_name'] = variant_map['trip_short_name'].apply(sorted)  # sort for consistency

# Convert to dictionary: route -> list of full variant strings (e.g., "129A")
route_variants = {}
for _, row in variant_map.iterrows():
    route = row['route_id']
    short_names = row['trip_short_name']
    route_variants[route] = [f"{route}{sn}" for sn in short_names]

# ----------------------------------------------------------------------
# Prepare df for expansion
# ----------------------------------------------------------------------
# Ensure Route is string (matches route_id type)
df['Route'] = df['Route'].astype(str).str.strip()

# Create a temporary column with the list of variants for each row
df['_variants'] = df['Route'].map(route_variants)

# Keep only rows where we have variants (others will have NaN and be dropped)
df_with_variants = df.dropna(subset=['_variants']).copy()

# Explode the list to create one row per variant
df_expanded = df_with_variants.explode('_variants').reset_index(drop=True)

# Replace the original Route column with the variant string
df_expanded['Route'] = df_expanded['_variants']

# Drop the temporary column
df_expanded = df_expanded.drop(columns=['_variants'])

# If you want to keep rows without variants (i.e., routes that have no trip_short_name),
# you can concat them back. Here we drop them (optional).
# df_final = pd.concat([df_expanded, df[df['_variants'].isna()].drop(columns=['_variants'])], ignore_index=True)

# For now, assign back to df (or create a new variable)
df = df_expanded

In [12]:
import os

# ----------------------------------------------------------------------
# Aggregate data by Route, Year, Transit, Incident_Category
# ----------------------------------------------------------------------
# Ensure Route is integer (clean) and Year is present
df['Route'] = pd.to_numeric(df['Route'], errors='coerce').astype('Int64')
df['Year'] = df['Year'].astype('Int64')

# Group to get counts and total delay minutes
route_agg = df.groupby(['Route', 'Year', 'Transit', 'Incident_Category'], as_index=False).agg(
    Delay_Count=('Min Delay', 'count'),
    Total_Delay_Min=('Min Delay', 'sum')
)

# ----------------------------------------------------------------------
# Compute active_in_2025 (route had >20 incidents in 2025)
# ----------------------------------------------------------------------
# Total incidents per route in 2025 (all categories)
incidents_2025 = df[df['Year'] == 2025].groupby('Route').size().reset_index(name='total_2025')
active_routes = incidents_2025[incidents_2025['total_2025'] > 20]['Route'].tolist()

# Add active_in_2025 flag to aggregated data
route_agg['active_in_2025'] = route_agg['Route'].isin(active_routes)

# ----------------------------------------------------------------------
# Add route_long_name (constant per route)
# ----------------------------------------------------------------------
# Get unique route_long_name per Route (take first non-null)
route_names = df.dropna(subset=['route_long_name']).groupby('Route')['route_long_name'].first().reset_index()
route_agg = route_agg.merge(route_names, on='Route', how='left')

# If some routes still missing long name, fill with placeholder
route_agg['route_long_name'] = route_agg['route_long_name'].fillna('Unknown')

# ----------------------------------------------------------------------
# Save to assets/data/route_analysis.csv
# ----------------------------------------------------------------------
output_dir = os.path.join('assets', 'data')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'route_analysis.csv')
route_agg = route_agg[route_agg['Route']!=0]
route_agg.to_csv(output_path, index=False)

print(f"✅ route_analysis.csv saved to {output_path}")
print(f"Shape: {route_agg.shape}")
print(route_agg.head())

✅ route_analysis.csv saved to assets\data\route_analysis.csv
Shape: (12953, 8)
   Route  Year Transit Incident_Category  Delay_Count  Total_Delay_Min  \
0      1  2014  Subway          Cleaning           55            245.0   
1      1  2014  Subway          External          244           2436.0   
2      1  2014  Subway           General          301           1378.0   
3      1  2014  Subway    Infrastructure          275           2752.0   
4      1  2014  Subway        Management           27            141.0   

   active_in_2025        route_long_name  
0            True  Yonge-University Line  
1            True  Yonge-University Line  
2            True  Yonge-University Line  
3            True  Yonge-University Line  
4            True  Yonge-University Line  


In [13]:
route_agg

,Route,Year,Transit,Incident_Category,Delay_Count,Total_Delay_Min,active_in_2025,route_long_name
0,1,2014,Subway,Cleaning,55,245.0,True,Yonge-University Line
1,1,2014,Subway,External,244,2436.0,True,Yonge-University Line
2,1,2014,Subway,General,301,1378.0,True,Yonge-University Line
3,1,2014,Subway,Infrastructure,275,2752.0,True,Yonge-University Line
4,1,2014,Subway,Management,27,141.0,True,Yonge-University Line
...,...,...,...,...,...,...,...,...
12948,999,2024,Bus,Collision,6,32.0,True,Unknown
12949,999,2024,Bus,Mechanical,17,151.0,True,Unknown
12950,999,2024,Bus,Operations,23,155.0,True,Unknown
12951,999,2024,Bus,Passenger,4,54.0,True,Unknown


In [13]:
import requests
import zipfile
import pandas as pd
import re
from io import BytesIO, StringIO
from collections import defaultdict
from rapidfuzz import fuzz, process

# ----------------------------------------------------------------------
# 1. Download stops.txt
# ----------------------------------------------------------------------
def download_stops_gtfs():
    """Download stops.txt from the TTC GTFS package and return as DataFrame."""
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"

    # Get package metadata
    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # Find the ZIP resource
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # Extract stops.txt
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        if 'stops.txt' in zf.namelist():
            with zf.open('stops.txt') as f:
                stops_df = pd.read_csv(f)
            print("✅ Extracted stops.txt")
        else:
            raise Exception("stops.txt not found in ZIP")

    return stops_df

# Download stops
stops = download_stops_gtfs()
print(f"Stops loaded: {len(stops)} rows")

# ----------------------------------------------------------------------
# 2. Prepare stop matching structures
# ----------------------------------------------------------------------
def normalize_stop(name):
    if pd.isna(name):
        return ''
    s = name.lower()
    s = re.sub(r'[^\w\s]', ' ', s)      # punctuation -> space
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def clean_location(loc):
    if pd.isna(loc):
        return ''
    s = loc.lower()
    s = re.sub(r'\(.*?\)', '', s)       # remove parenthetical notes
    s = re.sub(r'[^\w\s]', ' ', s)      # punctuation -> space
    s = re.sub(r'\s+', ' ', s).strip()
    # Expand common direction abbreviations
    s = re.sub(r'\bw\b', ' west', s)
    s = re.sub(r'\be\b', ' east', s)
    s = re.sub(r'\bn\b', ' north', s)
    s = re.sub(r'\bs\b', ' south', s)
    return s

def prepare_stop_matcher(stops_df):
    stops = stops_df.copy()
    stops['norm_stop'] = stops['stop_name'].apply(normalize_stop)
    stops['stop_words'] = stops['norm_stop'].apply(lambda x: set(x.split()))

    stop_names = stops['stop_name'].tolist()
    stop_norms = stops['norm_stop'].tolist()
    stop_words_list = stops['stop_words'].tolist()
    stop_lats = stops['stop_lat'].tolist()
    stop_lons = stops['stop_lon'].tolist()

    word_to_indices = defaultdict(set)
    for idx, words in enumerate(stop_words_list):
        for word in words:
            word_to_indices[word].add(idx)

    return (stop_names, stop_norms, stop_words_list, stop_lats, stop_lons, word_to_indices)

def match_location(loc, stop_names, stop_norms, stop_words_list, stop_lats, stop_lons, word_to_indices):
    if pd.isna(loc):
        return None, None, None

    cleaned = clean_location(loc)
    words = set(cleaned.split())

    # Candidate stops that contain any of the words
    candidates = set()
    for w in words:
        candidates.update(word_to_indices.get(w, set()))
    if not candidates:
        return None, None, None

    # 1. Exact subset match
    for idx in candidates:
        if words.issubset(stop_words_list[idx]):
            return stop_names[idx], stop_lats[idx], stop_lons[idx]

    # 2. Substring match
    for idx in candidates:
        if cleaned in stop_norms[idx]:
            return stop_names[idx], stop_lats[idx], stop_lons[idx]

    # 3. Fuzzy match
    cand_indices = list(candidates)
    cand_norms = [stop_norms[i] for i in cand_indices]
    result = process.extractOne(
        cleaned,
        cand_norms,
        scorer=fuzz.token_set_ratio,
        score_cutoff=60,
        processor=None
    )
    if result:
        best_pos = cand_norms.index(result[0])
        best_idx = cand_indices[best_pos]
        return stop_names[best_idx], stop_lats[best_idx], stop_lons[best_idx]

    return None, None, None

# Build matcher data
matcher_data = prepare_stop_matcher(stops)

# ----------------------------------------------------------------------
# 3. Apply matching to df['Location'] (with progress bar)
# ----------------------------------------------------------------------
# If you have tqdm installed, uncomment the next line for a progress bar:
# from tqdm import tqdm
# tqdm.pandas()
# matched = df['Location'].progress_apply(lambda loc: pd.Series(match_location(loc, *matcher_data)))

matched = df['Location'].apply(lambda loc: pd.Series(match_location(loc, *matcher_data)))
matched.columns = ['stop_name', 'stop_lat', 'stop_lon']

# Add to original dataframe
df = pd.concat([df, matched], axis=1)

# ----------------------------------------------------------------------
# 4. Quick check
# ----------------------------------------------------------------------
print("✅ Stop columns added.")
print("Sample matches (first 10):")
print(df[['Location', 'stop_name', 'stop_lat', 'stop_lon']].head(10))
print(f"Matched rows: {df['stop_name'].notna().sum()} / {len(df)} ({df['stop_name'].notna().mean()*100:.1f}%)")

📥 Downloading GTFS ZIP from: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/b811ead4-6eaf-4adb-8408-d389fb5a069c/resource/c920e221-7a1c-488b-8c5b-6d8cd4e85eaf/download/completegtfs.zip
✅ Extracted stops.txt
Stops loaded: 9417 rows
✅ Stop columns added.
Sample matches (first 10):
                Location                                          stop_name  \
0     YORK MILLS STATION  Yonge St at York Mills Rd North Side - York Mi...   
1   ENTIRE RUN FOR ROUTE                                                NaN   
2    LAWRENCE AND WARDEN                    Warden Ave at Lawrence Ave East   
3        KIPLING STATION  Kipling Ave at Belfield Rd - Etobicoke North G...   
4       VP AND ELLESMERE                         Morrish Rd at Ellesmere Rd   
5       SCARBOROUGH TOWN                    Walmart Scarborough Town Centre   
6  HUMBERLINE AT WOODLOT     Humberline Dr at Woodlot Cres (West) East Side   
7   PASSMORE AND MARKHAM                         Markham Rd at Passmore Ave   


In [14]:
df_location = df[df['stop_name'].notnull()]

In [15]:
df_location

,Date,Time,Day,Location,Incident,Min Delay,Min Gap,Route,Direction,Vehicle,Transit,Year,Month,Weekday,Hour,Incident_Category,route_long_name,stop_name,stop_lat,stop_lon
0,2014-01-01,00:23:00,Wednesday,YORK MILLS STATION,Mechanical,10.0,20.0,95,E,1734.0,Bus,2014,1,Wednesday,0.0,Mechanical,York Mills,Yonge St at York Mills Rd North Side - York Mi...,43.744863,-79.406660
2,2014-01-01,01:28:00,Wednesday,LAWRENCE AND WARDEN,Mechanical,10.0,20.0,54,WB,7478.0,Bus,2014,1,Wednesday,1.0,Mechanical,Lawrence East,Warden Ave at Lawrence Ave East,43.745560,-79.294980
3,2014-01-01,01:30:00,Wednesday,KIPLING STATION,Emergency Services,18.0,36.0,112,N,8084.0,Bus,2014,1,Wednesday,1.0,Passenger,West Mall,Kipling Ave at Belfield Rd - Etobicoke North G...,43.705019,-79.563774
4,2014-01-01,01:37:00,Wednesday,VP AND ELLESMERE,Investigation,10.0,20.0,24,n,7843.0,Bus,2014,1,Wednesday,1.0,Passenger,Victoria Park,Morrish Rd at Ellesmere Rd,43.790410,-79.173512
5,2014-01-01,01:50:00,Wednesday,SCARBOROUGH TOWN,Mechanical,10.0,20.0,129,N,1755.0,Bus,2014,1,Wednesday,1.0,Mechanical,McCowan North,Walmart Scarborough Town Centre,43.777570,-79.258592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
973803,2025-12-31,17:27,Wednesday,SIGNET ARROW STOP,EXSA,6.0,12.0,6,E,6502.0,LRT,2025,12,Wednesday,17.0,Mechanical,Finch West Line,Signet Arrow Station Eastbound Platform,43.753302,-79.535930
973804,2025-12-31,18:19,Wednesday,MARTIN GROVE STOP,EXAC,4.0,14.0,6,W,6512.0,LRT,2025,12,Wednesday,18.0,Mechanical,Finch West Line,Bloor St West at Martin Grove Rd,43.639772,-79.544930
973805,2025-12-31,19:09,Wednesday,SIGNET ARROW STOP,MXWEA,4.0,14.0,6,E,6513.0,LRT,2025,12,Wednesday,19.0,External,Finch West Line,Signet Arrow Station Eastbound Platform,43.753302,-79.535930
973806,2025-12-31,19:11,Wednesday,EMERY STOP,MXD,4.0,14.0,6,W,6501.0,LRT,2025,12,Wednesday,19.0,Passenger,Finch West Line,Emery Station Eastbound Platform,43.752088,-79.542106


In [16]:
import os
import pandas as pd

# Assuming df_location is already loaded with required columns:
# stop_name, stop_lat, stop_lon, Year, Transit, Incident_Category, Min Delay

# ----------------------------------------------------------------------
# Aggregate by stop, year, transit, incident category
# ----------------------------------------------------------------------
# Ensure Year is integer (optional)
df_location['Year'] = pd.to_numeric(df_location['Year'], errors='coerce').astype('Int64')

# Group and aggregate
location_agg = df_location.groupby(
    ['stop_name', 'stop_lat', 'stop_lon', 'Year', 'Transit', 'Incident_Category'],
    as_index=False
).agg(
    Delay_Count=('Min Delay', 'count'),
    Total_Delay_Min=('Min Delay', 'sum')
)

# Optionally, remove rows where stop_name is null (if you only want matched locations)
# location_agg = location_agg.dropna(subset=['stop_name'])

# ----------------------------------------------------------------------
# Save to assets/data/location_analysis.csv
# ----------------------------------------------------------------------
output_dir = os.path.join('assets', 'data')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'location_analysis.csv')
location_agg.to_csv(output_path, index=False)

print(f"✅ location_analysis.csv saved to {output_path}")
print(f"Shape: {location_agg.shape}")
print(location_agg.head())

✅ location_analysis.csv saved to assets\data\location_analysis.csv
Shape: (138627, 8)
                         stop_name   stop_lat   stop_lon  Year    Transit  \
0  1 Front St West - Union Station  43.646205 -79.377816  2016  Streetcar   
1  1 Front St West - Union Station  43.646205 -79.377816  2016  Streetcar   
2  1 Front St West - Union Station  43.646205 -79.377816  2019        Bus   
3  1 Front St West - Union Station  43.646205 -79.377816  2019  Streetcar   
4   1 and 3 Concorde Pl North Side  43.728143 -79.328163  2014        Bus   

  Incident_Category  Delay_Count  Total_Delay_Min  
0        Mechanical            4             35.0  
1        Operations            1             15.0  
2           General            1             20.0  
3        Mechanical            1              1.0  
4        Scheduling            1             30.0  


In [17]:
output_path = os.path.join(output_dir, 'full_delay_data.csv')
df.to_csv(output_path, index=False)

In [2]:
import pandas as pd
df = pd.read_csv(r'C:\Users\bains\OneDrive\Documents\VS Code\Toronto-Transit-Delays\assets\data\full_delay_data.csv')
df

,Date,Time,Day,Location,Incident,Min Delay,Min Gap,Route,Direction,Vehicle,Transit,Year,Month,Weekday,Hour,Incident_Category,route_long_name,stop_name,stop_lat,stop_lon
0,2014-01-01,00:23:00,Wednesday,YORK MILLS STATION,Mechanical,10.0,20.0,95,E,1734.0,Bus,2014,1,Wednesday,0.0,Mechanical,York Mills,Yonge St at York Mills Rd North Side - York Mi...,43.744863,-79.406660
1,2014-01-01,00:55:00,Wednesday,ENTIRE RUN FOR ROUTE,General Delay,33.0,66.0,102,b/w,8110.0,Bus,2014,1,Wednesday,0.0,General,Markham Rd,NaN,NaN,NaN
2,2014-01-01,01:28:00,Wednesday,LAWRENCE AND WARDEN,Mechanical,10.0,20.0,54,WB,7478.0,Bus,2014,1,Wednesday,1.0,Mechanical,Lawrence East,Warden Ave at Lawrence Ave East,43.745560,-79.294980
3,2014-01-01,01:30:00,Wednesday,KIPLING STATION,Emergency Services,18.0,36.0,112,N,8084.0,Bus,2014,1,Wednesday,1.0,Passenger,West Mall,Kipling Ave at Belfield Rd - Etobicoke North G...,43.705019,-79.563774
4,2014-01-01,01:37:00,Wednesday,VP AND ELLESMERE,Investigation,10.0,20.0,24,n,7843.0,Bus,2014,1,Wednesday,1.0,Passenger,Victoria Park,Morrish Rd at Ellesmere Rd,43.790410,-79.173512
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
973803,2025-12-31,17:27,Wednesday,SIGNET ARROW STOP,EXSA,6.0,12.0,6,E,6502.0,LRT,2025,12,Wednesday,17.0,Mechanical,Finch West Line,Signet Arrow Station Eastbound Platform,43.753302,-79.535930
973804,2025-12-31,18:19,Wednesday,MARTIN GROVE STOP,EXAC,4.0,14.0,6,W,6512.0,LRT,2025,12,Wednesday,18.0,Mechanical,Finch West Line,Bloor St West at Martin Grove Rd,43.639772,-79.544930
973805,2025-12-31,19:09,Wednesday,SIGNET ARROW STOP,MXWEA,4.0,14.0,6,E,6513.0,LRT,2025,12,Wednesday,19.0,External,Finch West Line,Signet Arrow Station Eastbound Platform,43.753302,-79.535930
973806,2025-12-31,19:11,Wednesday,EMERY STOP,MXD,4.0,14.0,6,W,6501.0,LRT,2025,12,Wednesday,19.0,Passenger,Finch West Line,Emery Station Eastbound Platform,43.752088,-79.542106


In [9]:
df['Incident_Category'].value_counts()

Incident_Category
Mechanical        303625
Operations        220783
General           162748
Passenger         155492
Scheduling         73096
Cleaning           25115
Collision          17776
Infrastructure     10244
External            4043
Management           886
Name: count, dtype: int64

In [6]:
a= df[df['Transit']=='Bus']
a['Incident'].unique()

array([310, 305, 312, 505, 501, 510, 509, 504, 511, 503, 506, 512, 306,
       507, 508, 301, 304], dtype=int64)

In [18]:
# Ensure Route is numeric (coerce errors to NaN)
df['Route'] = pd.to_numeric(df['Route'], errors='coerce')

# Filter rows where Route is between 300-399 or 500-599
mask = ((df['Route'] >= 300) & (df['Route'] <= 399)) | ((df['Route'] >= 500) & (df['Route'] <= 599))
filtered = df.loc[mask]

# Get unique route numbers and their first associated long name
unique_routes = filtered[['Route', 'route_long_name']].drop_duplicates().sort_values('Route').reset_index(drop=True)

print(unique_routes)

    Route      route_long_name
0     300       Bloor-Danforth
1     301                Queen
2     302                  NaN
3     303                  NaN
4     304                 King
5     305               Dundas
6     306              Carlton
7     307             Bathurst
8     308                  NaN
9     309                  NaN
10    310              Spadina
11    311                  NaN
12    312             St Clair
13    313                  NaN
14    315   Evans-Brown's Line
15    316  Kingston Rd-McCowan
16    317                  NaN
17    319                  NaN
18    320                Yonge
19    321                  NaN
20    322              Coxwell
21    324        Victoria Park
22    325            Don Mills
23    329             Dufferin
24    332                  NaN
25    334             Eglinton
26    335                 Jane
27    336           Finch West
28    337            Islington
29    339           Finch East
30    341                Keele
31    34

In [24]:
#!/usr/bin/env python3
"""
prepare_data.py

Reads location_analysis.csv and the GeoJSON boundary files from assets/data/,
performs spatial joins, and writes aggregated JSON files:
- wards_aggregated.json
- neighbourhoods_aggregated.json
- hotspots_aggregated.json

These files are optimised for client‑side filtering in the TTC Delay Analytics app.
"""

import os
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
DATA_DIR = "assets/data"          # relative to where the script is run
LOCATION_CSV = os.path.join(DATA_DIR, "location_analysis.csv")
WARD_GEOJSON = os.path.join(DATA_DIR, "gtawards.geojson")
NEIGHBOURHOOD_GEOJSON = os.path.join(DATA_DIR, "toronto.geojson")

OUT_WARDS = os.path.join(DATA_DIR, "wards_aggregated.json")
OUT_NEIGHBOURHOODS = os.path.join(DATA_DIR, "neighbourhoods_aggregated.json")
OUT_HOTSPOTS = os.path.join(DATA_DIR, "hotspots_aggregated.json")

# ----------------------------------------------------------------------
# Load location data
# ----------------------------------------------------------------------
print("📥 Loading location data...")
df = pd.read_csv(LOCATION_CSV)

# Ensure numeric columns are proper numbers
df['stop_lat'] = pd.to_numeric(df['stop_lat'], errors='coerce')
df['stop_lon'] = pd.to_numeric(df['stop_lon'], errors='coerce')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')   # allows NaN if missing
df['Delay_Count'] = pd.to_numeric(df['Delay_Count'], errors='coerce').fillna(0).astype(int)
df['Total_Delay_Min'] = pd.to_numeric(df['Total_Delay_Min'], errors='coerce').fillna(0)

# Drop rows with invalid coordinates
df = df.dropna(subset=['stop_lat', 'stop_lon'])
print(f"   Loaded {len(df)} records with valid coordinates.")

# Create geometry column for spatial joins
geometry = [Point(xy) for xy in zip(df['stop_lon'], df['stop_lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# ----------------------------------------------------------------------
# Wards aggregation
# ----------------------------------------------------------------------
print("\n🏛️ Processing wards...")
wards_gdf = gpd.read_file(WARD_GEOJSON)

# Ensure both are in the same CRS (they are, but just in case)
if wards_gdf.crs != gdf.crs:
    wards_gdf = wards_gdf.to_crs(gdf.crs)

# Spatial join: assign ward name to each point (inner join – keep only points inside a ward)
joined_wards = gpd.sjoin(gdf, wards_gdf[['AREA_NAME', 'geometry']], how='inner', predicate='within')

# Group by ward, year, transit, incident category
wards_agg = joined_wards.groupby(['AREA_NAME', 'Year', 'Transit', 'Incident_Category']).agg({
    'Delay_Count': 'sum',
    'Total_Delay_Min': 'sum'
}).reset_index()

# Rename columns to match frontend expectations
wards_agg.rename(columns={
    'AREA_NAME': 'ward',
    'Year': 'year',
    'Transit': 'transit',
    'Incident_Category': 'category',
    'Delay_Count': 'delay_count',
    'Total_Delay_Min': 'total_delay_min'
}, inplace=True)

# Replace NaN years (if any) with null – they will be filtered out later anyway
wards_agg['year'] = wards_agg['year'].where(pd.notna(wards_agg['year']), None)

print(f"   Created {len(wards_agg)} aggregated ward records.")

# Write to JSON
wards_agg.to_json(OUT_WARDS, orient='records', indent=None)   # compact JSON
print(f"   ✅ Saved to {OUT_WARDS}")

# ----------------------------------------------------------------------
# Neighbourhoods aggregation
# ----------------------------------------------------------------------
print("\n🏘️ Processing neighbourhoods...")
neighbourhoods_gdf = gpd.read_file(NEIGHBOURHOOD_GEOJSON)

if neighbourhoods_gdf.crs != gdf.crs:
    neighbourhoods_gdf = neighbourhoods_gdf.to_crs(gdf.crs)

joined_neigh = gpd.sjoin(gdf, neighbourhoods_gdf[['AREA_NAME', 'geometry']], how='inner', predicate='within')

neigh_agg = joined_neigh.groupby(['AREA_NAME', 'Year', 'Transit', 'Incident_Category']).agg({
    'Delay_Count': 'sum',
    'Total_Delay_Min': 'sum'
}).reset_index()

neigh_agg.rename(columns={
    'AREA_NAME': 'neighbourhood',
    'Year': 'year',
    'Transit': 'transit',
    'Incident_Category': 'category',
    'Delay_Count': 'delay_count',
    'Total_Delay_Min': 'total_delay_min'
}, inplace=True)

neigh_agg['year'] = neigh_agg['year'].where(pd.notna(neigh_agg['year']), None)

print(f"   Created {len(neigh_agg)} aggregated neighbourhood records.")
neigh_agg.to_json(OUT_NEIGHBOURHOODS, orient='records', indent=None)
print(f"   ✅ Saved to {OUT_NEIGHBOURHOODS}")

# ----------------------------------------------------------------------
# Hotspots aggregation (by unique stop)
# ----------------------------------------------------------------------
print("\n🔥 Processing hotspots (stops)...")
# Group by stop coordinates, year, transit, category
hotspot_agg = df.groupby(['stop_lat', 'stop_lon', 'Year', 'Transit', 'Incident_Category']).agg({
    'Delay_Count': 'sum',
    'Total_Delay_Min': 'sum'
}).reset_index()

hotspot_agg.rename(columns={
    'stop_lat': 'lat',
    'stop_lon': 'lon',
    'Year': 'year',
    'Transit': 'transit',
    'Incident_Category': 'category',
    'Delay_Count': 'delay_count',
    'Total_Delay_Min': 'total_delay_min'
}, inplace=True)

hotspot_agg['year'] = hotspot_agg['year'].where(pd.notna(hotspot_agg['year']), None)

print(f"   Created {len(hotspot_agg)} aggregated hotspot records.")
hotspot_agg.to_json(OUT_HOTSPOTS, orient='records', indent=None)
print(f"   ✅ Saved to {OUT_HOTSPOTS}")

print("\n🎉 All aggregations complete!")

📥 Loading location data...
   Loaded 138878 records with valid coordinates.

🏛️ Processing wards...
   Created 5445 aggregated ward records.
   ✅ Saved to assets/data\wards_aggregated.json

🏘️ Processing neighbourhoods...
   Created 17356 aggregated neighbourhood records.
   ✅ Saved to assets/data\neighbourhoods_aggregated.json

🔥 Processing hotspots (stops)...
   Created 138867 aggregated hotspot records.
   ✅ Saved to assets/data\hotspots_aggregated.json

🎉 All aggregations complete!


In [22]:
import requests
import pandas as pd
import zipfile
import json
import os
from io import BytesIO, StringIO

def generate_route_geometries():
    """
    Downloads trips.txt and shapes.txt from the merged GTFS package,
    and generates a JSON file mapping route+short_name combinations
    to their shape geometries (list of [lat, lon] points).
    The file is saved as assets/data/route_geometries.json.
    """
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"
    output_dir = os.path.join("assets", "data")
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "route_geometries.json")

    print("\n🗺️ Generating route geometries from GTFS...")

    # 1. Get package metadata
    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # 2. Find the ZIP resource (non-datastore, format=ZIP)
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # 3. Extract trips.txt and shapes.txt into memory
    required_files = ['trips.txt', 'shapes.txt']
    gtfs_data = {}
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        for fname in required_files:
            if fname in zf.namelist():
                with zf.open(fname) as f:
                    gtfs_data[fname] = f.read().decode('utf-8')
                print(f"✅ Extracted {fname}")
            else:
                print(f"⚠️ {fname} not found in ZIP")
                return

    # 4. Read trips.txt
    trips = pd.read_csv(StringIO(gtfs_data['trips.txt']))
    # Keep only necessary columns: route_id, trip_short_name, shape_id
    trips = trips[['route_id', 'trip_short_name', 'shape_id']].drop_duplicates()
    # Convert to string and fill missing short names
    trips['route_id'] = trips['route_id'].astype(str)
    trips['trip_short_name'] = trips['trip_short_name'].fillna('').astype(str).str.strip()

    # 5. Read shapes.txt
    shapes = pd.read_csv(StringIO(gtfs_data['shapes.txt']))
    shapes = shapes[['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence']]
    shapes = shapes.sort_values(['shape_id', 'shape_pt_sequence'])

    # 6. Build geometry per shape_id
    shape_geometries = {}
    for shape_id, group in shapes.groupby('shape_id'):
        # Create list of [lat, lon] pairs in order
        coords = group[['shape_pt_lat', 'shape_pt_lon']].values.tolist()
        shape_geometries[shape_id] = coords

    # 7. Merge trips with shape geometries
    # For each (route_id, trip_short_name) we need a geometry.
    # There might be multiple trips with same route+short_name but different shape_id.
    # We'll take the first shape_id for each combination (assuming consistency).
    route_geometries = {}
    # Group trips by (route_id, trip_short_name) and take first shape_id
    for (route_id, short_name), group in trips.groupby(['route_id', 'trip_short_name']):
        shape_id = group.iloc[0]['shape_id']  # first shape_id
        if shape_id in shape_geometries:
            key = route_id + short_name  # e.g., "100A"
            route_geometries[key] = shape_geometries[shape_id]
        else:
            print(f"⚠️ Shape ID {shape_id} not found for route {key}")

    # 8. Save to JSON
    with open(output_file, 'w') as f:
        json.dump(route_geometries, f, indent=2)
    print(f"✅ Route geometries saved to {output_file}")
    return route_geometries

# ----------------------------------------------------------------------
# Example usage
# ----------------------------------------------------------------------
if __name__ == "__main__":
    generate_route_geometries()


🗺️ Generating route geometries from GTFS...
📥 Downloading GTFS ZIP from: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/b811ead4-6eaf-4adb-8408-d389fb5a069c/resource/c920e221-7a1c-488b-8c5b-6d8cd4e85eaf/download/completegtfs.zip
✅ Extracted trips.txt
✅ Extracted shapes.txt


C:\Users\bains\AppData\Local\Temp\ipykernel_4064\1775921265.py:61: DtypeWarning: Columns (0: trip_short_name, 1: shape_id) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv(StringIO(gtfs_data['trips.txt']))


⚠️ Shape ID nan not found for route 49S
✅ Route geometries saved to assets\data\route_geometries.json
